**Objective:** Load the raw F42119 sales order history extract, inspect its structure, filter relevant columns, clean the data, and save a clean version ready for EDA and modelling.

**Source Table:** F42119 — Sales Order Detail History (JD Edwards EnterpriseOne)  
**Period:** January 2023 – December 2025  
**Currency:** Laos Kip (LAK)  
**Target Variables:**
- `Quantity Shipped` → Monthly total units sold
- `Extended Price` → Monthly total revenue (LAK)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('Libraries loaded successfully ✅')

In [ ]:
# Load raw data
df_raw = pd.read_csv('raw_data.csv', low_memory=False)

print('Raw data loaded successfully')
print(f'Rows    : {df_raw.shape[0]:,}')
print(f'Columns : {df_raw.shape[1]:,}')

From 268 columns, we keep only what we need for time series forecasting.

| Column Name | JDE Field | Purpose |
|---|---|---|
| `G/L Date` | SDDGL | Time axis for forecasting |
| `Quantity Shipped` | SDQTYS | Target variable 1 — units sold |
| `Extended Price` | SDAEXP | Target variable 2 — revenue (LAK) |
| `Ln Ty` | SDLNTY | Filter: keep Stock lines only (S) |
| `Short Item No` | SDITM | Item-level granularity |
| `Description` | SDDESC | Item description (human readable) |
| `Business Unit` | SDMCU | Branch/Plant |
| `Order Number` | SDDOCO | Order reference |

In [ ]:
# Select only relevant columns
COLS = [
    'G/L Date',
    'Quantity Shipped',
    'Extended Price',
    'Ln Ty',
    'Short Item No',
    'Description ',       # Note: JDE exports sometimes add trailing space
    'Business Unit',
    'Order Number'
]

# Strip whitespace from all column names first
df_raw.columns = df_raw.columns.str.strip()

# Redefine after stripping
COLS = [
    'G/L Date',
    'Quantity Shipped',
    'Extended Price',
    'Ln Ty',
    'Short Item No',
    'Description',
    'Business Unit',
    'Order Number'
]

df = df_raw[COLS].copy()

# Rename for easier use throughout the project
df.columns = [
    'gl_date',
    'qty_shipped',
    'extended_price',
    'line_type',
    'item_no',
    'description',
    'business_unit',
    'order_number'
]

print(f'✅ Selected {len(COLS)} columns')
df.head(5)

In [ ]:
print(df.dtypes)
print()
print(df.isnull().sum())
print()
print(df[['qty_shipped', 'extended_price']].describe())

In [ ]:
# Just remove the rows where extended_price is null.
df = df[df['extended_price'].notna()].reset_index(drop=True)

### Convert GL Date to Datetime datatype

In [ ]:
# Convert gl_date to datetime safely
df['gl_date'] = pd.to_datetime(df['gl_date'], errors='coerce')

# Check if any dates failed to parse (would be NaT)
bad_dates = df['gl_date'].isna().sum()
print(f'Failed date parses : {bad_dates}')
print(f'Date range         : {df["gl_date"].min().date()} → {df["gl_date"].max().date()}')
print(f'Total months       : {(df["gl_date"].max().year - df["gl_date"].min().year) * 12 + df["gl_date"].max().month - df["gl_date"].min().month + 1}')

In [ ]:
df.dtypes

We apply the same filters you would use in a JDE Sales Analysis report:

- `line_type == 'S'` → Keep Stock lines only (exclude freight, text, non-stock)
- `qty_shipped > 0` → Keep shipped lines only (exclude cancellations and reversals)
- `extended_price > 0` → Keep positive revenue lines only

In [ ]:
# Before we proceed for the above step, let's convert the qty_shipped and extended_price to FLOAT

df['qty_shipped'] = pd.to_numeric(df['qty_shipped'].astype(str).str.replace(',', ''), errors='coerce')
df['extended_price'] = pd.to_numeric(df['extended_price'].astype(str).str.replace(',', ''), errors='coerce')

print('✅ Converted to numeric')
print(f'qty_shipped dtype    : {df["qty_shipped"].dtype}')
print(f'extended_price dtype : {df["extended_price"].dtype}')

In [ ]:
rows_before = len(df)

# Filter 1: Stock lines only
df = df[df['line_type'].str.strip() == 'S']
print(f'After line_type = S filter  : {len(df):,} rows (removed {rows_before - len(df):,})')

# Filter 2: Shipped quantity > 0
df = df[df['qty_shipped'] > 0]
print(f'After qty_shipped > 0 filter: {len(df):,} rows (removed {rows_before - len(df):,} total)')

# Filter 3: Positive revenue only
df = df[df['extended_price'] > 0]
print(f'After extended_price > 0    : {len(df):,} rows (removed {rows_before - len(df):,} total)')

# Filter 4: Drop rows where GL date is null
df = df[df['gl_date'].notna()]
print(f'After dropping null dates   : {len(df):,} rows')

print(f'\n✅ Final clean rows: {len(df):,}')

### Confirm the Date Range (Validation)

In [ ]:
# Check the Min and Max Date
print(f'Earliest GL Date : {df["gl_date"].min().date()}')
print(f'Latest GL Date   : {df["gl_date"].max().date()}')

# Check for any data outside your specific range (Jan 2023 - Dec 2025)
out_of_range = df[(df['gl_date'] < '2023-01-01') | (df['gl_date'] > '2025-12-31')]
if len(out_of_range) > 0:
    print(f'⚠️  {len(out_of_range)} rows outside Jan 2023–Dec 2025 — removing them')
    df = df[(df['gl_date'] >= '2023-01-01') & (df['gl_date'] <= '2025-12-31')]
else:
    print('✅ All dates within Jan 2023 – Dec 2025')

print(f'\nFinal row count: {len(df):,}')

### Extract Time Columns

In [ ]:
df['year']       = df['gl_date'].dt.year
df['month']      = df['gl_date'].dt.month
df['month_name'] = df['gl_date'].dt.strftime('%b')
df['quarter']    = df['gl_date'].dt.quarter
df['year_month'] = df['gl_date'].dt.to_period('M')

print('✅ Time columns added')
df[['gl_date', 'year', 'month', 'month_name', 'quarter', 'year_month']].head(5)

### Final Sanity Checks

In [ ]:
print('=== DISTINCT UNIQUE VALUES ===')
print(f'Unique Items         : {df["item_no"].nunique():,}')
print(f'Unique Business Units: {df["business_unit"].nunique():,}')
print(f'Unique Orders        : {df["order_number"].nunique():,}')
print(f'Line Type values     : {df["line_type"].unique()}')
print()

print('=== REVENUE SUMMARY (LAK) ===')
print(f'Total Revenue  : LAK {df["extended_price"].sum():,.0f}')
print(f'Total Qty      : {df["qty_shipped"].sum():,.0f} units')
print(f'Avg Unit Price : LAK {(df["extended_price"].sum() / df["qty_shipped"].sum()):,.2f}')
print()

print('=== TOP 5 ITEMS BY REVENUE ===')
top_items = df.groupby('description')['extended_price'].sum().sort_values(ascending=False).head(5)
print(top_items.apply(lambda x: f'LAK {x:,.0f}'))

### Monthly Aggregation

Aggregate daily transaction records to **monthly level** — this is what we will feed into Prophet and LSTM.

Each row = one month, with total quantity shipped and total revenue.

In [ ]:
# Aggregate to monthly
monthly = df.groupby('year_month').agg(
    total_qty     = ('qty_shipped', 'sum'),
    total_revenue = ('extended_price', 'sum'),
    order_count   = ('order_number', 'nunique')
).reset_index()

# Convert period back to timestamp for time series compatibility
monthly['ds'] = monthly['year_month'].dt.to_timestamp()

# Sort by date
monthly = monthly.sort_values('ds').reset_index(drop=True)

print(f'✅ Monthly aggregation complete')
print(f'   Total months: {len(monthly)}')
print(f'   Expected    : 36 months (Jan 2023 – Dec 2025)')
print()
monthly

- groupby('year_month') → groups all rows by month (e.g. all rows from Jan 2023 together, Feb 2023 together, etc.)
- .agg(...) → for each month group, calculate:

- total_qty → sum of all qty_shipped (total units sold that month)
- total_revenue → sum of all extended_price (total LAK revenue that month)
- order_count → count distinct order numbers (how many unique orders that month)

reset_index() → brings year_month back as a regular column

### Check for Missing Months

Time series models require a continuous sequence. If any month has zero transactions, we need to fill it.

In [ ]:
# Generate full month range
full_range = pd.date_range(start='2023-01-01', end='2025-12-01', freq='MS')

# Check which months are missing
existing_months = set(monthly['ds'])
missing_months  = [m for m in full_range if m not in existing_months]

missing_months

In [ ]:
if missing_months:
    print(f'⚠️  Missing months found: {len(missing_months)}')
    for m in missing_months:
        print(f'   → {m.strftime("%Y-%m")}')

    # Fill missing months with 0
    missing_df = pd.DataFrame({
        'ds'           : missing_months,
        'total_qty'    : 0,
        'total_revenue': 0,
        'order_count'  : 0
    })
    monthly = pd.concat([monthly[['ds','total_qty','total_revenue','order_count']], missing_df])
    monthly = monthly.sort_values('ds').reset_index(drop=True)
    print(f'✅ Missing months filled with 0')
else:
    print(f'✅ No missing months — all 36 months present')

print(f'\nFinal monthly rows: {len(monthly)}')

In [ ]:
monthly

### Visualization

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('F42119 — Monthly Sales Overview (Jan 2023 – Dec 2025)', fontsize=14, fontweight='bold')

# Plot 1: Quantity
axes[0].plot(monthly['ds'], monthly['total_qty'], color='steelblue', linewidth=2, marker='o', markersize=4)
axes[0].fill_between(monthly['ds'], monthly['total_qty'], alpha=0.1, color='steelblue')
axes[0].set_title('Monthly Quantity Shipped (Units)')
axes[0].set_ylabel('Quantity (Units)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
axes[0].grid(True, alpha=0.3)

# Plot 2: Revenue
axes[1].plot(monthly['ds'], monthly['total_revenue'], color='darkorange', linewidth=2, marker='o', markersize=4)
axes[1].fill_between(monthly['ds'], monthly['total_revenue'], alpha=0.1, color='darkorange')
axes[1].set_title('Monthly Revenue (LAK)')
axes[1].set_ylabel('Revenue (LAK)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:,.1f}B'))
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('reports', exist_ok=True)
plt.savefig('reports/monthly_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved to reports/monthly_overview.png')

### Save the Cleaned Data

In [ ]:
os.makedirs('data/processed', exist_ok=True)

# Save cleaned transaction-level data
df.to_csv('data/processed/cleaned_transactions.csv', index=False)
print(f'✅ Cleaned transactions saved → data/processed/cleaned_transactions.csv ({len(df):,} rows)')

# Save monthly aggregated data (this is what Prophet and LSTM will use)
monthly.to_csv('data/processed/monthly_aggregated.csv', index=False)
print(f'✅ Monthly aggregated data saved → data/processed/monthly_aggregated.csv ({len(monthly)} rows)')

In [ ]:
print(monthly[['ds', 'total_qty', 'total_revenue', 'order_count']].to_string())